<a href="https://colab.research.google.com/github/Devil-92/100_Days_of_ML/blob/main/GDA_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Discriminant Analysis from Scratch (LDA & QDA)

NumPy implementation of Gaussian Discriminant Analysis, evaluated on the load_breast_cancer dataset.

## Idea

Model each class as a Gaussian and classify with Bayes' rule:

$$\hat{y} = \arg\max_c \; \log \pi_c - \tfrac{1}{2}\log|\Sigma_c| - \tfrac{1}{2}(x-\mu_c)^\top \Sigma_c^{-1}(x-\mu_c)$$

- **QDA**: each class has its own covariance $\Sigma_c$, so the decision boundary is quadratic.
- **LDA**: all classes share one pooled covariance $\Sigma = \sum_c \pi_c \hat{\Sigma}_c$. The quadratic terms cancel, so the boundary is linear.



In [1]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target

In [2]:
X.shape

(569, 30)

In [3]:
y.shape

(569,)

In [4]:
import numpy as np

class GDA :

  def __init__(self , shared = False) :
    self.shared = shared

  def fit(self , X , y) :
    X = np.asarray(X , float)
    y = np.asarray(y)

    self.classes = np.unique(y)
    print(f'Total Number of Classes : {self.classes.size}')
    self.pi = np.array([(y == c).mean() for c in self.classes])
    print(f'Shape of pi : {self.pi.shape}')
    self.mu = np.array([X[y == c].mean(axis = 0) for c in self.classes])
    print(f'Shape of mu : {self.mu.shape}')
    self.cov = np.array([np.cov(X[y == c].T , bias = True) for c in self.classes])
    print(f'Shape of Cov Matrix : {self.cov.shape}')

    if self.shared == True :
      S = sum(p * C for p , C in zip(self.pi , self.cov))
      self.cov = np.array([S] * len(self.classes))
      print(f'Shape of Shared Cov Matrix : {self.cov.shape}')

  def predict(self , X) :
    X = np.asarray(X , float)

    scores = []

    for p , m , c in zip(self.pi , self.mu , self.cov) :
      d = X - m
      maha = np.diag(d @ np.linalg.inv(c) @ d.T)
      logdet = np.log(np.linalg.det(c))
      scores.append(np.log(p) - 0.5 * (maha + logdet))

    return self.classes[np.argmax(scores , axis = 0)]

In [5]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42 , stratify = y)

In [6]:
model_QDA = GDA()

In [7]:
model_QDA.fit(X_train , y_train)

Total Number of Classes : 2
Shape of pi : (2,)
Shape of mu : (2, 30)
Shape of Cov Matrix : (2, 30, 30)


In [8]:
y_pred = model_QDA.predict(X_test)

In [9]:
print((y_test == y_pred).mean())

0.9473684210526315


In [10]:
model_LDA = GDA(shared = True)

In [11]:
model_LDA.fit(X_train , y_train)

Total Number of Classes : 2
Shape of pi : (2,)
Shape of mu : (2, 30)
Shape of Cov Matrix : (2, 30, 30)
Shape of Shared Cov Matrix : (2, 30, 30)


In [12]:
y_pred = model_LDA.predict(X_test)

In [13]:
print((y_test == y_pred).mean())

0.956140350877193


In [14]:
print('LDA Performs well than QDA here')

LDA Performs well than QDA here
